In [6]:
import time
import json
import websocket
import socket
import numpy as np
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

# --- CONFIGURARE LAPTOP ---
USE_SIMULATOR = False # Asigură-te că e FALSE pentru casca reală!
SERIAL_PORT = "COM10"
SERVER_URI = "ws://127.0.0.1:65432" # PUNE IP-UL PC-ULUI AICI!

def main():
    BoardShim.enable_dev_board_logger()
    params = BrainFlowInputParams()
    
    board_id = BoardIds.SYNTHETIC_BOARD if USE_SIMULATOR else BoardIds.CYTON_DAISY_BOARD
    if not USE_SIMULATOR: 
        params.serial_port = SERIAL_PORT
    
    board = BoardShim(board_id, params)
    ws = None

    try:
        board.prepare_session()
        board.start_stream()
        print("✅ Stream pornit. Asteptam stabilizare semnal (3s)...")
        time.sleep(3)

        # Citim exact 16 canale
        eeg_channels = BoardShim.get_eeg_channels(board_id)[:16]
        print(f"ℹ️ Canale EEG transmise: {len(eeg_channels)}")

        ws = websocket.WebSocket()
        print(f"🔌 Conectare la {SERVER_URI}...")
        ws.connect(SERVER_URI, timeout=10)
        print("✅ CONECTAT LA SERVERUL DE INFERENȚĂ!")

        while True:
            data = board.get_board_data()
            
            # Așteptăm să strângem un mic calup
            if data.shape[1] < 10:
                time.sleep(0.05)
                continue

            # Extragem DOAR canalele EEG brute. 
            # FĂRĂ FILTRARE AICI, FĂRĂ DOWNSAMPLING!
            eeg_data = data[eeg_channels, :]
            
            try:
                ws.send(json.dumps(eeg_data.tolist()))
                
                # Prevenim blocajul buffer-ului TCP
                if ws.sock:
                    ws.sock.settimeout(0.001)
                    while True: 
                        ws.recv()
            except (websocket.WebSocketTimeoutException, socket.timeout):
                pass # E normal, continuăm
            except Exception as e:
                print(f"⚠️ Eroare rețea: {e}")
                try:
                    ws.close()
                    time.sleep(1)
                    ws.connect(SERVER_URI, timeout=10)
                    print("🔄 Reconectat!")
                except: pass

            time.sleep(0.05)

    except KeyboardInterrupt:
        print("\n🛑 Oprire manuala.")
    finally:
        if ws: ws.close()
        if board.is_prepared(): 
            board.release_session()
            print("🔌 Sesiune hardware închisă.")

if __name__ == "__main__":
    main()

BrainFlowError: BOARD_WRITE_ERROR:4 unable to prepare streaming session